In [26]:
"""
Deepseek paper. 
"""

'\nDeepseek paper. \n'

In [28]:
from data import load_data, batches

tokenizer, train_ids, val_ids = load_data()

In [29]:
text = "Once upon a time, there was a little dog bro."
ids = tokenizer.encode(text).ids

print(ids)

[16600, 4465, 260, 1014, 14, 1031, 515, 260, 2961, 6397, 3151, 16]


In [30]:
tokenizer.get_vocab_size()

129280

In [31]:
for x, y in batches(train_ids, block_size=512, batch_size=128):
    print(x.shape, y.shape)
    break

torch.Size([128, 512]) torch.Size([128, 512])


In [32]:
import math

import torch
from torch import nn
from torch.nn import functional as F

class KV_Cache():
    def __init__(self, n_layers):
        # kv_cache: n_layers * (k, v): (k, v) of shape (B, N, S, H)
        self.kv_cache = [None] * n_layers

    def __getitem__(self, i):
        return self.kv_cache[i]

    def __setitem__(self, i, kv):
        self.kv_cache[i] = kv

    def seq_len(self):
        return 0 if self.kv_cache[0] is None else self.kv_cache[0][0].shape[2]


class Attention(nn.Module):
    def __init__(self, n_heads, d_emb):
        super().__init__()
        self.n_heads = n_heads
        self.d_emb = d_emb
        self.d_head = d_emb // n_heads
        self.o_proj = nn.Linear(d_emb, d_emb)

    def split_heads(self, x):
        # inp: B, s, d
        # out: B, n_head, s, d // n_heads

        x = x.view(x.shape[0], x.shape[1], self.n_heads, -1) # B, Block, N, emb // N
        return x.transpose(1,2)

    def merge_heads(self, x):
        # inp: B, N, S, D
        # out: B, S, D
        x = x.transpose(1, 2).contiguous()
        return x.view(x.shape[0], -1, self.d_emb)


    def forward(self, q, k, v, causal=True, kv_cache=None, layer_idx=None):
        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v) # B, N, S, D

        if kv_cache is not None:
            if kv_cache[layer_idx] is not None:
                k = torch.cat([kv_cache[layer_idx][0], k], dim=-2)
                v = torch.cat([kv_cache[layer_idx][1], v], dim=-2)

            kv_cache[layer_idx] = (k, v)

        # q: B, N, S, D @ B, N, D, S -> B, N, S, S
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)

        if causal:
            S_q, S_k = q.shape[-2], k.shape[-2]
            # with a kv cache S_k > S_q, so the diagonal slides right by the offset:
            # query i sits at absolute position (S_k - S_q) + i and may see every j <= that
            mask = torch.ones(S_q, S_k, dtype=torch.bool, device=q.device).tril(diagonal=S_k - S_q)
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)

        # B, N, S, S @ B, N, S, D -> B, N, S, D
        out = weights @ v

        return self.o_proj(self.merge_heads(out))

class MLP(nn.Module):
    def __init__(self, d_emb):
        super().__init__()
        # proj to 4 * D
        self.up_proj = nn.Linear(d_emb, 4 * d_emb)
        self.act = nn.GELU()
        self.down_proj = nn.Linear(4 * d_emb, d_emb)

    def forward(self, x):
        # x: B, S, D
        x = self.up_proj(x)
        x = self.act(x)
        x = self.down_proj(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, n_heads, d_emb, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.att = Attention(n_heads=n_heads, d_emb=d_emb)
        self.mlp = MLP(d_emb)
        self.ln1 = nn.LayerNorm(d_emb)
        self.ln2 = nn.LayerNorm(d_emb)
        self.ln_mem = nn.LayerNorm(d_emb)

        self.q_proj = nn.Linear(d_emb, d_emb)
        self.k_proj = nn.Linear(d_emb, d_emb)
        self.v_proj = nn.Linear(d_emb, d_emb)

    def forward(self, x, memory=None, kv_cache=None):

        h = self.ln1(x)

        kv_src = h if memory is None else self.ln_mem(memory)

        q = self.q_proj(h)
        k = self.k_proj(kv_src)
        v = self.v_proj(kv_src)

        x = x + self.att(q, k, v, kv_cache=kv_cache, layer_idx=self.layer_idx)
        x = x + self.mlp(self.ln2(x))

        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_emb, n_heads, n_layers, max_seqlen):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(vocab_size, d_emb)
        self.pe = nn.Embedding(max_seqlen, d_emb)
        n_enc = n_layers // 2
        self.encoder = nn.ModuleList([TransformerBlock(n_heads, d_emb, layer_idx=i) for i in range(n_enc)])
        # offset the decoder indices so the two halves don't share cache slots
        self.decoder = nn.ModuleList([TransformerBlock(n_heads, d_emb, layer_idx=n_enc + i) for i in range(n_layers - n_enc)])

        self.ln_f = nn.LayerNorm(d_emb)
        self.final_emb = nn.Linear(d_emb, vocab_size)

    def forward(self, input_ids, kv_cache=None):
        start = 0 if kv_cache is None else kv_cache.seq_len()

        # input_ids = [B, S]
        input_idx = torch.arange(start, start + input_ids.shape[1], device=input_ids.device)
        x = self.emb(input_ids) + self.pe(input_idx)

        for block in self.encoder:
            x = block(x, kv_cache=kv_cache)

        memory = x
        for block in self.decoder:
            x = block(x, memory=memory, kv_cache=kv_cache)

        return self.final_emb(self.ln_f(x))

    @torch.no_grad()
    def prefill(self, input_ids):
        kv_cache = KV_Cache(self.n_layers)
        logits = self(input_ids, kv_cache)
        return logits[:, -1], kv_cache

    @torch.no_grad()
    def decode_step(self, input_ids, kv_cache=None):
        logits = self.forward(input_ids[:, None], kv_cache=kv_cache)
        return logits[:, -1], kv_cache

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temp=1.0):
        logits, kv_cache = self.prefill(input_ids)
        out = [input_ids]
        for _ in range(max_new_tokens):
            if temp == 0:
                next_ids = logits.argmax(dim=-1)
            else:
                next_ids = torch.multinomial((logits / temp).softmax(-1), num_samples=1)[:, 0]

            out.append(next_ids[:, None])
            logits, kv_cache = self.decode_step(next_ids, kv_cache=kv_cache)

        return torch.cat(out, dim=1)

In [33]:
# The DeepSeek tokenizer carries a 128k vocab but IMDB only touches 46,601 ids.
# At d_emb=256 the embedding + output projection is 95% of the model, so remapping
# to the ids that actually occur buys ~3.7x more steps per minute at ~0.4% token loss.
keep_k = 32000

counts = torch.bincount(train_ids, minlength=tokenizer.get_vocab_size())
keep_ids = counts.argsort(descending=True)[:keep_k].sort().values   # original ids, ascending
unk_id = keep_k                                                     # everything else lands here

old2new = torch.full((tokenizer.get_vocab_size(),), unk_id, dtype=torch.long)
old2new[keep_ids] = torch.arange(keep_k)

vocab_size   = keep_k + 1
train_stream = old2new[train_ids]
val_stream   = old2new[val_ids]


def decode(ids):
    """Map compact ids back to the real tokenizer and render them as text."""
    ids = ids.detach().cpu().clone()
    ids[ids == unk_id] = 0          # nothing sensible to show for unk
    return tokenizer.decode(keep_ids[ids].tolist())


kept = (train_stream != unk_id).float().mean()
print(f"vocab {tokenizer.get_vocab_size():,} -> {vocab_size:,} | {kept:.2%} of train tokens survive")

vocab 129,280 -> 32,001 | 99.59% of train tokens survive


In [34]:
import math
import time

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)

# falls back to the raw 128k vocab if the remap cell above wasn't run
vocab_size   = globals().get("vocab_size", tokenizer.get_vocab_size())
train_stream = globals().get("train_stream", train_ids)
val_stream   = globals().get("val_stream", val_ids)

block_size  = 256
batch_size  = 16
d_emb       = 256
n_heads     = 4
n_layers    = 4      # split in half -> 2 encoder + 2 decoder
lr          = 3e-4
warmup      = 100
max_steps   = 1150   # measured: ~5 min on mps at 32k vocab (throughput sags as it heats up)
max_minutes = 5      # hard stop, in case a step is slower than expected
eval_every  = 250
eval_iters  = 10

model = Transformer(
    vocab_size=vocab_size,
    d_emb=d_emb,
    n_heads=n_heads,
    n_layers=n_layers,
    max_seqlen=block_size,
).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params on {device}, vocab {vocab_size:,}")

steps_per_epoch = (train_stream.numel() - 1) // block_size // batch_size
total_steps = min(max_steps, steps_per_epoch)


def lr_at(step):
    if step < warmup:
        return lr * (step + 1) / warmup
    # cosine decay from lr down to lr/10
    p = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return lr / 10 + 0.5 * (lr - lr / 10) * (1 + math.cos(math.pi * p))


def loss_on(x, y):
    logits = model(x)                                   # B, S, V
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.reshape(-1))


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for i, (x, y) in enumerate(batches(val_stream, block_size, batch_size, device=device, shuffle=False)):
        if i >= eval_iters:
            break
        losses.append(loss_on(x, y).item())
    model.train()
    return sum(losses) / len(losses)


step = 0
t0 = time.time()
model.train()
for x, y in batches(train_stream, block_size, batch_size, device=device):
    if step >= total_steps or time.time() - t0 > max_minutes * 60:
        break

    for group in opt.param_groups:
        group["lr"] = lr_at(step)

    loss = loss_on(x, y)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    if step % 50 == 0:
        elapsed = time.time() - t0
        print(f"step {step:5d}/{total_steps} | loss {loss.item():.4f} | lr {lr_at(step):.2e} "
              f"| {(step + 1) * batch_size * block_size / elapsed:,.0f} tok/s | {elapsed / 60:.1f} min")
    if step % eval_every == 0 and step > 0:
        print(f"  -> val loss {evaluate():.4f}")

    step += 1

print(f"done: {step} steps in {(time.time() - t0) / 60:.1f} min | val loss {evaluate():.4f}")

19.6M params on mps, vocab 32,001
step     0/1150 | loss 10.5478 | lr 3.00e-06 | 6,592 tok/s | 0.0 min
step    50/1150 | loss 9.0341 | lr 1.53e-04 | 17,764 tok/s | 0.2 min
step   100/1150 | loss 7.1711 | lr 3.00e-04 | 18,088 tok/s | 0.4 min
step   150/1150 | loss 6.6763 | lr 2.98e-04 | 18,163 tok/s | 0.6 min
step   200/1150 | loss 6.7220 | lr 2.94e-04 | 18,190 tok/s | 0.8 min
step   250/1150 | loss 6.3822 | lr 2.87e-04 | 18,210 tok/s | 0.9 min
  -> val loss 6.6156
step   300/1150 | loss 6.3393 | lr 2.77e-04 | 18,104 tok/s | 1.1 min
step   350/1150 | loss 6.2999 | lr 2.64e-04 | 18,170 tok/s | 1.3 min
step   400/1150 | loss 6.3833 | lr 2.49e-04 | 18,212 tok/s | 1.5 min
step   450/1150 | loss 6.2465 | lr 2.32e-04 | 18,199 tok/s | 1.7 min
step   500/1150 | loss 6.1178 | lr 2.14e-04 | 18,108 tok/s | 1.9 min
  -> val loss 6.2676
step   550/1150 | loss 6.4544 | lr 1.95e-04 | 17,751 tok/s | 2.1 min
step   600/1150 | loss 5.9693 | lr 1.75e-04 | 17,346 tok/s | 2.4 min
step   650/1150 | loss 5.97